# Simulación del estómago — Fase 6: modelo integrado

**Un único bucle de tiempo** que avanza a la vez:
- la **química local**: pH en cada celda, con tampón en equilibrio y activación de la pepsina;
- el **transporte** del contenido: mezcla por las ondas, flujo neto hacia el píloro y entrada del jugo por las paredes;
- las **partículas de comida**: arrastre, ablandamiento por el ácido *que tienen alrededor*, rotura en el antro y erosión por la pepsina local, que libera péptidos al líquido;
- el **píloro abierto**: el líquido sale y las partículas pequeñas pasan con él, mientras que las grandes vuelven al antro;
- el **volumen**, que cambia por la secreción y el vaciado.

**Y el flujo, ¿cómo entra?** Las dos escalas de tiempo son muy distintas. Una onda antral dura 20 s y el LBM necesita unos 7000 pasos para resolverla. La digestión dura horas: resolverla entera con LBM serían ~5 millones de pasos, unas 7 h de CPU. Usamos el método estándar para este caso, el **promediado en fase** (acoplamiento multiescala):
1. **Escala rápida**: el LBM de la fase 3 resuelve una onda completa. Con trazadores medimos qué hace el flujo en promedio por onda: cuánto dispersa el líquido en cada zona, $D(x)$; cómo se mueven las partículas que sedimentan; y con qué frecuencia las atrapa una contracción terminal, $\lambda_g(x)$.
2. **Escala lenta**: el bucle integrado usa esos mapas en lugar de la velocidad instantánea, con pasos de ~1 s en lugar de ~3 ms.

La sección 3 verifica que el promediado reproduce la dispersión del LBM directo.

**Comparación final.** Una tabla y una figura ponen las cuatro versiones del modelo (1a, 1b, 4 y la integrada) frente a los mismos datos clínicos, con los tres tipos de error.

**Tiempo en Colab**: unos 8 minutos en CPU. Las celdas más lentas, la del conjunto de incertidumbre y la de la animación, tienen un interruptor para acortarlas.

## 1. Arquitectura del modelo

In [ ]:
import math, time, os, glob, pickle
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Patch
from scipy.stats import qmc
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
    "animation.embed_limit": 60,
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"
COL_V = {"1a": C5, "1b": C3, "4": C1, "6": C2}
os.makedirs("figuras", exist_ok=True)
VERIFICACION = []

def caja(ax, x, y, w, h, texto, color, fs=9.5, negrita=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=color, ec="none", alpha=0.18))
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc="none", ec=color, lw=1.4))
    ax.text(x + w / 2, y + h / 2, texto, ha="center", va="center", fontsize=fs, color=GRIS, weight="bold" if negrita else "normal", wrap=True)

def flecha(ax, a, b, texto="", color=GRIS, curva=0.0, fs=8.5):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle="-|>", mutation_scale=14, color=color, lw=1.3,
                                 connectionstyle=f"arc3,rad={curva}"))
    if texto:
        ax.text((a[0] + b[0]) / 2, (a[1] + b[1]) / 2 + 0.12, texto, ha="center", fontsize=fs, color=TINTA2, style="italic")

fig, ax = plt.subplots(figsize=(14, 6.2)); ax.set_xlim(0, 14); ax.set_ylim(0, 6.6); ax.axis("off")
caja(ax, 0.2, 4.6, 3.6, 1.6, "ESCALA RÁPIDA (1 onda = 20 s)\nLattice Boltzmann, paredes móviles\n(fase 3)", C5, negrita=True)
caja(ax, 4.6, 4.6, 3.8, 1.6, "Mapas promediados en fase\n• dispersión del líquido D(x)\n• molienda λg(x) y dispersión de partículas", C4)
flecha(ax, (3.8, 5.4), (4.6, 5.4), "trazadores")
ax.text(9.0, 6.3, "ESCALA LENTA: un único bucle de tiempo (Δt ≈ 1 s, 4 h)", fontsize=11, weight="bold", color=GRIS)
caja(ax, 9.0, 4.4, 2.3, 1.5, "Transporte del líquido\n(dispersión + flujo neto\nal píloro + jugo)", C1)
caja(ax, 11.6, 4.4, 2.2, 1.5, "Química local\npH(x,y), pepsina\n(fase 1b en cada celda)", C3)
caja(ax, 9.0, 1.8, 2.3, 1.7, "Partículas lagrangianas\nrotura · erosión ·\nablandamiento (fase 4)", C2)
caja(ax, 11.6, 1.8, 2.2, 1.7, "Píloro abierto\nsale el líquido;\nsolo partículas < dc", GRIS_MEDIO)
caja(ax, 4.6, 1.8, 3.8, 1.7, "Volumen V(t)\nsecreción − vaciado\n(regla energética, fase 1b)", C4)
flecha(ax, (8.4, 5.2), (9.0, 5.2), "D(x)")
flecha(ax, (8.4, 4.8), (9.0, 2.9), "λg(x)", curva=-0.2)
flecha(ax, (11.3, 5.15), (11.6, 5.15))
flecha(ax, (12.4, 4.4), (11.0, 3.5))
ax.text(12.55, 3.85, "pH y pepsina\nlocales", fontsize=8.5, color=TINTA2, style="italic")
flecha(ax, (9.6, 3.5), (9.6, 4.4))
ax.text(9.72, 3.85, "péptidos\ndisueltos", fontsize=8.5, color=TINTA2, style="italic", ha="left")
flecha(ax, (11.3, 2.6), (11.6, 2.6), "")
flecha(ax, (8.4, 2.6), (9.0, 2.6), "flujo neto")
flecha(ax, (8.4, 3.1), (9.0, 4.6), "", curva=0.2)
ax.text(0.3, 3.4, "Qué aporta cada fase", fontsize=10.5, weight="bold", color=GRIS)
for k, (lin, col) in enumerate([("1a  química 0D (ácido, pepsina)", COL_V["1a"]), ("1b  + vaciado, tampón en equilibrio", COL_V["1b"]),
                                ("2   transporte 2D (volúmenes finitos)", GRIS), ("3   flujo: Lattice Boltzmann", C5),
                                ("4   partículas + criba pilórica", COL_V["4"]), ("6   todo acoplado, química local, píloro abierto", COL_V["6"])]):
    ax.add_patch(plt.Circle((0.45, 3.0 - 0.42 * k), 0.09, color=col)); ax.text(0.7, 3.0 - 0.42 * k, lin, va="center", fontsize=9.5)
fig.savefig("figuras/00_arquitectura.png", dpi=200, bbox_inches="tight"); plt.show()

## 2. Código

### 2.1 Escala rápida: flujo LBM (fase 3) y trazadores

In [ ]:
CODIGO_LBM = r'''
# ===========================================================================
# Lattice Boltzmann D2Q9 escrito con funciones PURAS (sin modificar arrays en el sitio).
# Así el mismo código corre con NumPy, CuPy (GPU, misma API) o jax.numpy (GPU + jit).
# Unidades de red: dx = dt = 1, c_s^2 = 1/3, nu = (tau - 1/2)/3.
# Convenio de arrays: f tiene forma (9, ny, nx); la fila j = 0 es el fondo (y crece hacia arriba).
# ===========================================================================
CX  = [0, 1, 0, -1, 0, 1, -1, -1, 1]
CY  = [0, 0, 1, 0, -1, 1, 1, -1, -1]
W   = [4 / 9] + [1 / 9] * 4 + [1 / 36] * 4
OPP = [0, 3, 4, 1, 2, 7, 8, 5, 6]          # dirección opuesta


class Red:
    # Constantes D2Q9 ya convertidas al backend (xp) y al tipo de dato elegidos
    def __init__(self, xp, dtype="float64"):
        self.xp, self.dtype = xp, dtype
        self.cx = xp.asarray(CX, dtype=dtype).reshape(9, 1, 1)
        self.cy = xp.asarray(CY, dtype=dtype).reshape(9, 1, 1)
        self.w = xp.asarray(W, dtype=dtype).reshape(9, 1, 1)


def equilibrio(rho, ux, uy, R):
    cu = 3.0 * (R.cx * ux + R.cy * uy)
    usq = 1.5 * (ux * ux + uy * uy)
    return R.w * rho * (1.0 + cu + 0.5 * cu * cu - usq)


def macroscopicas(f, R, Fx=0.0, Fy=0.0):
    rho = f.sum(axis=0)
    jx = f[1] - f[3] + f[5] - f[6] - f[7] + f[8]        # sum_i f_i c_ix, escrito explícitamente (más rápido)
    jy = f[2] - f[4] + f[5] + f[6] - f[7] - f[8]
    ux = (jx + 0.5 * Fx) / rho                          # desplazamiento de Guo: u = (sum f c + F/2)/rho
    uy = (jy + 0.5 * Fy) / rho
    return rho, ux, uy


def colision_bgk(f, rho, ux, uy, tau, R, Fx=0.0, Fy=0.0):
    feq = equilibrio(rho, ux, uy, R)
    cu = R.cx * ux + R.cy * uy
    cF = R.cx * Fx + R.cy * Fy
    # término de fuerza de Guo (2002): segundo orden con fuerza volumétrica
    S = (1.0 - 0.5 / tau) * R.w * (3.0 * ((R.cx - ux) * Fx + (R.cy - uy) * Fy) + 9.0 * cu * cF)
    return f - (f - feq) / tau + S


def propagar(f, R):
    xp = R.xp
    return xp.stack([xp.roll(f[i], shift=(CY[i], CX[i]), axis=(0, 1)) for i in range(9)])


def rebote(fs, fpost, solido, uwx, uwy, rho, R):
    # Rebote a mitad de enlace con pared móvil (Ladd):
    # si la población i llega a un nodo fluido x desde un nodo sólido x - c_i, se sustituye por
    # f_i(x) = f*_opp(i)(x) + 6 w_i rho (c_i · u_w),  con u_w la velocidad de la pared en ese nodo sólido.
    xp = R.xp
    salida = []
    fluido = ~solido
    for i in range(9):
        if i == 0:
            salida.append(fs[0]); continue
        desde_solido = xp.roll(solido, shift=(CY[i], CX[i]), axis=(0, 1)) & fluido
        cu_w = 0.0
        if CX[i] != 0 and uwx is not None:
            cu_w = cu_w + CX[i] * xp.roll(uwx, shift=(CY[i], CX[i]), axis=(0, 1))
        if CY[i] != 0 and uwy is not None:
            cu_w = cu_w + CY[i] * xp.roll(uwy, shift=(CY[i], CX[i]), axis=(0, 1))
        rebotado = fpost[OPP[i]] + 6.0 * W[i] * rho * cu_w
        salida.append(xp.where(desde_solido, rebotado, fs[i]))
    return xp.stack(salida)


def rellenar(f, solido_viejo, solido_nuevo, rho, uwx, uwy, R):
    # Nodos que pasan de sólido a fluido ("nodos frescos") se inicializan con el equilibrio:
    # densidad = media de los vecinos fluidos, velocidad = la de la pared (Lallemand y Luo, 2003).
    xp = R.xp
    fresco = solido_viejo & ~solido_nuevo
    fluido_previo = (~solido_viejo).astype(rho.dtype)
    suma, n = 0.0 * rho, 0.0 * rho
    for i in range(1, 9):
        suma = suma + xp.roll(rho * fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
        n = n + xp.roll(fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
    rho_vec = xp.where(n > 0, suma / xp.maximum(n, 1.0), 1.0)
    cero = 0.0 * rho
    f_fresco = equilibrio(rho_vec, cero if uwx is None else uwx, cero if uwy is None else uwy, R)
    f = xp.where(fresco[None], f_fresco, f)
    # dentro del sólido guardamos el equilibrio en reposo, f_i = w_i (no interviene en la dinámica)
    return xp.where(solido_nuevo[None], R.w + 0.0 * f, f)


def crear_paso(R, tau, geometria, Fx=0.0, Fy=0.0):
    # Devuelve paso(f, t) -> f(t+1). geometria(t) -> (solido, uwx, uwy) en unidades de red.
    def paso(f, t):
        solido, uwx, uwy = geometria(t)
        rho, ux, uy = macroscopicas(f, R, Fx, Fy)
        fpost = colision_bgk(f, rho, ux, uy, tau, R, Fx, Fy)
        fs = propagar(fpost, R)
        fs = rebote(fs, fpost, solido, uwx, uwy, rho, R)
        solido2, uwx2, uwy2 = geometria(t + 1)
        return rellenar(fs, solido, solido2, rho, uwx2, uwy2, R)
    return paso


def tension_viscosa(f, tau, R, Fx=0.0, Fy=0.0):
    # Tensor de tensiones viscosas a partir de la parte de no equilibrio (sin derivar el campo):
    # sigma_ab = -(1 - 1/(2 tau)) sum_i (f_i - feq_i) c_ia c_ib    [unidades de red]
    rho, ux, uy = macroscopicas(f, R, Fx, Fy)
    fneq = f - equilibrio(rho, ux, uy, R)
    k = -(1.0 - 0.5 / tau)
    sxx = k * (fneq * R.cx * R.cx).sum(axis=0)
    syy = k * (fneq * R.cy * R.cy).sum(axis=0)
    sxy = k * (fneq * R.cx * R.cy).sum(axis=0)
    return sxx, syy, sxy


def geometria_fija(solido, R):
    xp = R.xp
    cero = xp.zeros(solido.shape, dtype=R.dtype)
    return lambda t: (solido, cero, cero)


# ---------------------------------------------------------------------------
# Paredes que se mueven: canal peristáltico periódico (test de Shapiro)
# ---------------------------------------------------------------------------
def geometria_peristaltica(nx, ny, a, phi, lam, c, R):
    # Semiancho H(x,t) = a (1 + phi sin(2 pi (x - c t)/lam)) alrededor del eje y = ny/2.
    # Las paredes se mueven solo en vertical (como en Shapiro et al., 1969).
    xp = R.xp
    x = xp.arange(nx, dtype=R.dtype)[None, :] + 0.5
    y = xp.arange(ny, dtype=R.dtype)[:, None] + 0.5
    yc = ny / 2.0
    k = 2 * math.pi / lam
    def geom(t):
        H = a * (1.0 + phi * xp.sin(k * (x - c * t)))
        dHdt = -a * phi * k * c * xp.cos(k * (x - c * t))
        abajo, arriba = y < yc - H, y > yc + H
        solido = abajo | arriba
        uwy = xp.where(abajo, -dHdt, 0.0 * y) + xp.where(arriba, dHdt, 0.0 * y)
        return solido, None, uwy                        # uwx = None: las paredes solo se mueven en vertical
    return geom


def integrar(paso, f, t0, n_pasos, R, usar_jax=False):
    # Bucle de tiempo. Con JAX se compila con lax.fori_loop (todo el bucle en la GPU).
    if usar_jax:
        from jax import lax
        f, _ = lax.fori_loop(0, n_pasos, lambda i, ft: (paso(ft[0], ft[1]), ft[1] + 1.0), (f, R.xp.asarray(t0, dtype=R.dtype)))
        return f
    for n in range(n_pasos):
        f = paso(f, t0 + n)
    return f

@dataclass
class ParamAntro:
    # Geometría y movimiento (unidades físicas: cm, s)
    Lx: float = 10.0          # cm   longitud (fondo -> píloro)
    H: float = 4.0            # cm   anchura en reposo
    margen: float = 1.0       # cm   holgura para que el fondo pueda dilatarse
    x_ini: float = 4.5        # cm   donde empiezan las ondas antrales
    x_plena: float = 6.5      # cm   donde alcanzan su amplitud máxima
    L_fondo: float = 3.5      # cm   zona proximal que se dilata para conservar el volumen
    A_max: float = 1.4        # cm   profundidad de la contracción en cada pared (70 % de oclusión)
    w: float = 0.5            # cm   semiancho de la onda
    c: float = 0.25           # cm/s velocidad de la onda peristáltica
    T: float = 20.0           # s    periodo (3 ondas por minuto)
    nu: float = 0.1           # cm²/s viscosidad cinemática (comida líquida espesa; agua = 0.01)
    rho: float = 1.0          # g/cm³
    # Discretización
    nx: int = 128             # nodos a lo largo (sin contar las paredes de los extremos)
    tau: float = 0.65


class Unidades:
    # Conversión física <-> red: dx fija la malla; dt sale de igualar la viscosidad.
    def __init__(self, pa):
        self.dx = pa.Lx / pa.nx                        # cm
        self.nu_red = (pa.tau - 0.5) / 3.0
        self.dt = self.nu_red * self.dx ** 2 / pa.nu    # s
        self.vel = self.dx / self.dt                    # (cm/s) por unidad de red
        self.tension = pa.rho * self.vel ** 2           # (dyn/cm² = 0.1 Pa) por unidad de red

    def resumen(self, pa):
        return {"dx [cm]": self.dx, "dt [s]": self.dt, "pasos por onda": pa.T / self.dt,
                "c onda [red]": pa.c / self.vel, "Re = c (H/2)/nu": pa.c * (pa.H / 2) / pa.nu,
                "nu [red]": self.nu_red}


def geometria_antro(pa, R):
    # Paredes superior e inferior con ondas que viajan hacia el píloro (x = Lx).
    # El fondo (x < L_fondo) se dilata lo justo para que el volumen total no cambie.
    # Extremos x = 0 (fondo) y x = Lx (píloro, cerrado en esta fase) son paredes fijas.
    xp = R.xp
    U = Unidades(pa)
    dx = U.dx
    ny = int(round((pa.H + 2 * pa.margen) / dx))
    nx = pa.nx + 2                                       # + una columna de pared en cada extremo
    x = (xp.arange(nx, dtype=R.dtype)[None, :] - 0.5) * dx          # cm (la columna 0 es pared)
    y = (xp.arange(ny, dtype=R.dtype)[:, None] + 0.5) * dx
    rampa = xp.clip((x - pa.x_ini) / (pa.x_plena - pa.x_ini), 0.0, 1.0)
    envolvente = pa.A_max * rampa * rampa * (3 - 2 * rampa)         # suave (smoothstep)
    ventana_fondo = xp.clip((pa.L_fondo + 1.0 - x) / 1.0, 0.0, 1.0) * (x > 0) * (x < pa.Lx)
    int_ventana = float(xp.sum(ventana_fondo)) * dx
    x0 = pa.x_ini - 3 * pa.w
    K = int(math.ceil((pa.Lx + 3 * pa.w - x0) / (pa.c * pa.T))) + 1
    extremos = (x <= 0) | (x >= pa.Lx)
    def ondas(t_fis):
        s = t_fis % pa.T
        d = 0.0 * x
        dd = 0.0 * x
        for k in range(K):
            xk = x0 + pa.c * (s + k * pa.T)
            g = xp.exp(-((x - xk) / pa.w) ** 2)
            d = d + g
            dd = dd + g * 2 * (x - xk) * pa.c / pa.w ** 2
        return envolvente * d, envolvente * dd                      # delta [cm], d delta/dt [cm/s]
    def geom(t):
        t_fis = t * U.dt
        delta, ddelta = ondas(t_fis)
        interior = (x > 0) & (x < pa.Lx)
        area = xp.sum(delta * interior) * dx                         # área contraída por pared
        darea = xp.sum(ddelta * interior) * dx
        d_f = area / int_ventana                                     # dilatación del fondo
        dd_f = darea / int_ventana
        yb = pa.margen + delta - ventana_fondo * d_f
        yt = pa.margen + pa.H - delta + ventana_fondo * d_f
        vb = ddelta - ventana_fondo * dd_f                           # cm/s
        vt = -ddelta + ventana_fondo * dd_f
        abajo, arriba = y < yb, y > yt
        solido = abajo | arriba | extremos
        uwy = (xp.where(abajo, vb, 0.0 * y) + xp.where(arriba, vt, 0.0 * y)) * ~extremos / U.vel
        return solido, None, uwy
    return geom, nx, ny, U

def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11


def simular_antro(pa, R, periodos=4, guardar_cada_s=0.25, n_particulas=3000, semilla=1,
                  usar_jax=False, informar=True):
    # Ejecuta el antro y devuelve instantáneas (en NumPy y unidades físicas) + trayectorias de trazadores.
    xp = R.xp
    geom, nx, ny, U = geometria_antro(pa, R)
    paso = crear_paso(R, pa.tau, geom)
    f = R.w * xp.ones((9, ny, nx), dtype=R.dtype)                     # reposo, rho = 1
    pasos_bloque = max(1, int(round(guardar_cada_s / U.dt)))
    n_bloques = int(round(periodos * pa.T / (pasos_bloque * U.dt)))
    if usar_jax:
        import jax
        avanzar = jax.jit(lambda f, t0: integrar(paso, f, t0, pasos_bloque, R, usar_jax=True))
    else:
        avanzar = lambda f, t0: integrar(paso, f, t0, pasos_bloque, R)
    # trazadores: repartidos al azar por el fluido inicial (coordenadas de red)
    rng = np.random.default_rng(semilla)
    s0 = a_numpy(geom(0)[0])
    cand = np.argwhere(~s0)
    elegido = cand[rng.choice(len(cand), n_particulas, replace=False)]
    py = elegido[:, 0] + rng.uniform(-0.5, 0.5, n_particulas)
    px = elegido[:, 1] + rng.uniform(-0.5, 0.5, n_particulas)
    hist = {k: [] for k in ("t", "ux", "uy", "rho", "solido", "tau_max", "px", "py", "masa")}
    def campos(f, t):
        rho, ux, uy = macroscopicas(f, R)
        sxx, syy, sxy = tension_viscosa(f, pa.tau, R)
        tmax = xp.sqrt(0.25 * (sxx - syy) ** 2 + sxy ** 2)            # cizalla máxima
        sol = geom(t)[0]
        return [a_numpy(v) for v in (rho, ux, uy, tmax, sol)]
    t_red = 0
    rho, ux, uy, tmax, sol = campos(f, t_red)
    t_ini = time.time()
    for b in range(n_bloques + 1):
        hist["t"].append(t_red * U.dt)
        hist["ux"].append(np.where(sol, np.nan, ux) * U.vel)
        hist["uy"].append(np.where(sol, np.nan, uy) * U.vel)
        hist["rho"].append(np.where(sol, np.nan, rho))
        hist["tau_max"].append(np.where(sol, np.nan, tmax) * U.tension / 10.0)   # Pa
        hist["solido"].append(sol)
        hist["px"].append(px.copy()); hist["py"].append(py.copy())
        hist["masa"].append(float(np.sum(np.where(sol, 0.0, rho))))
        if b == n_bloques:
            break
        f = avanzar(f, float(t_red))
        t_red += pasos_bloque
        rho2, ux2, uy2, tmax2, sol2 = campos(f, t_red)
        # trazadores: Heun con el campo al principio y al final del bloque (desplazamiento en nodos)
        n_p = pasos_bloque
        vx1 = interpolar_bilineal(np.where(sol, 0, ux), px, py); vy1 = interpolar_bilineal(np.where(sol, 0, uy), px, py)
        qx, qy = px + n_p * vx1, py + n_p * vy1
        vx2 = interpolar_bilineal(np.where(sol2, 0, ux2), qx, qy); vy2 = interpolar_bilineal(np.where(sol2, 0, uy2), qx, qy)
        nx_, ny_ = px + 0.5 * n_p * (vx1 + vx2), py + 0.5 * n_p * (vy1 + vy2)
        ii = np.clip(np.rint(nx_).astype(int), 0, nx - 1); jj = np.clip(np.rint(ny_).astype(int), 0, ny - 1)
        dentro = ~sol2[jj, ii]
        px = np.where(dentro, nx_, px); py = np.where(dentro, ny_, py)   # si caería en la pared, se queda
        rho, ux, uy, tmax, sol = rho2, ux2, uy2, tmax2, sol2
        if informar and (b + 1) % max(1, n_bloques // 8) == 0:
            print(f"  t = {t_red * U.dt:6.1f} s   ({time.time() - t_ini:5.1f} s de cálculo)")
    out = {k: np.array(v) for k, v in hist.items()}
    out["U"], out["nx"], out["ny"], out["pasos"] = U, nx, ny, t_red
    out["segundos"] = time.time() - t_ini
    return out



'''

In [ ]:
def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11

In [ ]:
class CampoPeriodico:
    # Campo de velocidades de un periodo de onda (fase 3), repetido periódicamente en el tiempo.
    def __init__(self, datos):
        self.t = datos["t"]; self.ux = datos["ux"]; self.uy = datos["uy"]; self.sol = datos["solido"]
        self.x = datos["x"]; self.y = datos["y"]; self.dx = float(datos["dx"]); self.T = float(datos["periodo"])
        self.nu = float(datos["nu"])
        self.tau = datos["tau_max"] if "tau_max" in datos else None
        self.nf = len(self.t) - 1                    # el último marco repite el primero (t = T)
        self.dtf = self.t[1] - self.t[0]
        # hueco local (cm) en cada columna y marco: nº de nodos fluidos × dx
        self.hueco = (~self.sol).sum(axis=1) * self.dx

    def _marco(self, t):
        s = (t % self.T) / self.dtf
        k = int(np.floor(s)) % self.nf
        return k, s - np.floor(s)

    def velocidad(self, t, px, py):
        k, a = self._marco(t)
        u0 = self._interp(self.ux[k], px, py); u1 = self._interp(self.ux[k + 1], px, py)
        v0 = self._interp(self.uy[k], px, py); v1 = self._interp(self.uy[k + 1], px, py)
        return (1 - a) * u0 + a * u1, (1 - a) * v0 + a * v1

    def _indices(self, px, py):
        fx = (px - self.x[0]) / self.dx; fy = (py - self.y[0]) / self.dx
        return fx, fy

    def _interp(self, campo, px, py):
        fx, fy = self._indices(px, py)
        return interpolar_bilineal(campo, fx, fy)

    def solido_en(self, t, px, py):
        k, a = self._marco(t)
        sol = self.sol[k if a < 0.5 else k + 1]
        fx, fy = self._indices(px, py)
        i = np.clip(np.rint(fx).astype(int), 0, sol.shape[1] - 1); j = np.clip(np.rint(fy).astype(int), 0, sol.shape[0] - 1)
        return sol[j, i], sol, i, j

    def hueco_en(self, t, px):
        k, _ = self._marco(t)
        fx = (px - self.x[0]) / self.dx
        i = np.clip(np.rint(fx).astype(int), 0, self.hueco.shape[1] - 1)
        return self.hueco[k, i]


def tiempo_relajacion(d_cm, rho_p, rho_f, nu, v_rel):
    # Arrastre de Stokes con corrección de Schiller–Naumann: tau = rho_p d² / (18 mu f(Re_p))
    mu = rho_f * nu
    Re_p = np.abs(v_rel) * d_cm / nu
    f = 1 + 0.15 * Re_p ** 0.687
    return rho_p * d_cm ** 2 / (18 * mu * f)


def seguir_particulas(campo, d_mm, rho_p, rho_f=1.0, g=981.0, t_final=200.0, dt=0.02, semilla=0,
                      x_zona=8.0, frac_cuello=0.6, guardar_cada=0.5):
    # Partículas con inercia y sedimentación (gravedad hacia -y) en el campo periódico.
    # Integración exponencial con el fluido congelado durante dt (exacta para arrastre lineal).
    # Evento de compresión: la partícula está en el antro terminal (x > x_zona) dentro de un cuello
    # con hueco < frac_cuello · H. Se cuenta una vez por paso de onda.
    rng = np.random.default_rng(semilla)
    d_mm = np.asarray(d_mm, float); n = d_mm.size; d_cm = d_mm / 10
    s0 = campo.sol[0]
    cand = np.argwhere(~s0)
    sel = cand[rng.choice(len(cand), n, replace=True)]
    py = campo.y[sel[:, 0]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    px = campo.x[sel[:, 1]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    vx, vy = campo.velocidad(0.0, px, py)
    flot = (1 - rho_f / rho_p) * g
    cuerpo = (campo.x > 3.8) & (campo.x < 4.4)                              # cuerpo gástrico: sin ondas ni dilatación
    H_ref = float(np.median(campo.hueco[:, cuerpo]))
    ultimo_evento = np.full(n, -1e9)
    eventos_x = []
    eventos = np.zeros(n)
    tiempo_zona = np.zeros(n)
    hist_x, hist_y, hist_t = [px.copy()], [py.copy()], [0.0]
    cada = max(1, int(round(guardar_cada / dt)))
    for paso in range(int(round(t_final / dt))):
        t = paso * dt
        ux, uy = campo.velocidad(t, px, py)
        rx, ry = vx - ux, vy - uy
        tau = tiempo_relajacion(d_cm, rho_p, rho_f, campo.nu, np.hypot(rx, ry))
        vtx, vty = 0.0, -tau * flot                                          # velocidad terminal relativa
        e = np.exp(-dt / tau)
        cx_, cy_ = vx - ux - vtx, vy - uy - vty
        nx_ = px + (ux + vtx) * dt + cx_ * tau * (1 - e)
        ny_ = py + (uy + vty) * dt + cy_ * tau * (1 - e)
        vx = ux + vtx + cx_ * e; vy = uy + vty + cy_ * e
        # paredes: si el centro entra en el sólido, se recoloca en el nodo fluido más cercano de su columna
        dentro, sol, i, j = campo.solido_en(t + dt, nx_, ny_)
        if dentro.any():
            for q in np.where(dentro)[0]:
                col = ~sol[:, i[q]]
                if col.any():
                    jj = np.where(col)[0]
                    jn = jj[np.argmin(np.abs(jj - j[q]))]
                    ny_[q] = campo.y[jn]
                else:
                    nx_[q], ny_[q] = px[q], py[q]
                vy[q] = 0.0
        px, py = nx_, ny_
        # eventos de compresión en el antro terminal
        en_zona = px > x_zona
        tiempo_zona += en_zona * dt
        cuello = en_zona & (campo.hueco_en(t + dt, px) < frac_cuello * H_ref)
        nuevo = cuello & (t - ultimo_evento > 0.5 * campo.T)
        eventos += nuevo; ultimo_evento = np.where(nuevo, t, ultimo_evento)
        eventos_x.extend(px[nuevo].tolist())
        if (paso + 1) % cada == 0:
            hist_x.append(px.copy()); hist_y.append(py.copy()); hist_t.append(t + dt)
    return {"x": np.array(hist_x), "y": np.array(hist_y), "t": np.array(hist_t), "d_mm": d_mm,
            "eventos_por_min": eventos / (t_final / 60.0), "frac_tiempo_zona": tiempo_zona / t_final,
            "eventos_x": np.array(eventos_x)}

In [ ]:
def mapas_escala_rapida(campo, ondas=6, n=1500, d_part=3.0, rho_part=1.03, ancho_bin=1.0, margen=1.0, semilla=0, ventana=2):
    # Escala rápida -> mapas promediados en fase para el bucle lento.
    # 1) Trazadores de fluido: dispersión axial y transversal D_x(x), D_y(x) y deriva media u(x).
    # 2) Partículas densas (sedimentan): dispersión axial en la capa del fondo y frecuencia de molienda λ_g(x).
    T = campo.T
    dt_g = 0.5                                                   # s entre posiciones guardadas
    paso_onda = int(round(T / dt_g))
    bordes = np.arange(0.0, 10.0 + 1e-9, ancho_bin); centros = 0.5 * (bordes[1:] + bordes[:-1]); nb = len(centros)
    res = {"x": centros, "bordes": bordes, "T": T}
    for tipo, d, rp in (("fluido", 0.01, 1.0), ("particula", d_part, rho_part)):
        L = seguir_particulas(campo, np.full(n, d), rp, t_final=ondas * T, dt=0.02, guardar_cada=dt_g, semilla=semilla)
        X, Y = L["x"], L["y"] - margen
        idx = np.arange(0, X.shape[0], paso_onda)
        dxs, dys, x0s = [], [], []
        for a, b in zip(idx[:-ventana], idx[ventana:]):            # incrementos de `ventana` ondas (deslizantes)
            dxs.append(X[b] - X[a]); dys.append(Y[b] - Y[a]); x0s.append(X[a])
        dxs, dys, x0s = np.concatenate(dxs), np.concatenate(dys), np.concatenate(x0s)
        cual = np.clip(np.digitize(x0s, bordes) - 1, 0, nb - 1)
        Tv = ventana * T / 60                                                   # min
        # D a partir del desplazamiento cuadrático medio (incluye la dispersión por cizalla de la deriva media)
        Dx = np.array([np.mean(dxs[cual == i] ** 2) / (2 * Tv) if np.sum(cual == i) > 10 else np.nan for i in range(nb)])
        Dy = np.array([np.mean(dys[cual == i] ** 2) / (2 * Tv) if np.sum(cual == i) > 10 else np.nan for i in range(nb)])
        ux = np.array([np.mean(dxs[cual == i]) / Tv if np.sum(cual == i) > 10 else np.nan for i in range(nb)])
        rellenar = lambda v: np.interp(centros, centros[np.isfinite(v)], v[np.isfinite(v)])
        res[tipo] = {"Dx": np.maximum(rellenar(Dx), 1e-3), "Dy": np.maximum(rellenar(Dy), 1e-3), "u_media": rellenar(ux),
                     "x_tray": X, "y_tray": Y, "t_tray": L["t"]}
        if tipo == "particula":
            # tiempo pasado en cada franja (min) y eventos de molienda en cada franja
            cual_t = np.clip(np.digitize(X[1:], bordes) - 1, 0, nb - 1)
            tiempo = np.array([np.sum(cual_t == i) for i in range(nb)]) * dt_g / 60
            ev = np.histogram(L["eventos_x"], bordes)[0]
            res[tipo]["lambda_g"] = np.where(tiempo > 0, ev / np.maximum(tiempo, 1e-9), 0.0)
            res[tipo]["altura_media"] = float(np.mean(Y[-1]))
    return res


def paseo_aleatorio(x0, Dx_tab, x_tab, T_min, n_ondas, sub=40, semilla=0, L=10.0):
    # Paseo aleatorio de Itô con D(x) variable: dx = D'(x) dt + sqrt(2 D(x) dt) ξ  (forma conservativa), reflexión en 0 y L
    rng = np.random.default_rng(semilla)
    dD = np.gradient(Dx_tab, x_tab)
    x = x0.copy(); dt = T_min / sub
    salida = [x.copy()]
    for _ in range(n_ondas):
        for _ in range(sub):
            D = np.interp(x, x_tab, Dx_tab); D1 = np.interp(x, x_tab, dD)
            x = x + D1 * dt + np.sqrt(2 * D * dt) * rng.standard_normal(x.size)
            x = np.where(x < 0, -x, x); x = np.where(x > L, 2 * L - x, x)
        salida.append(x.copy())
    return np.array(salida)

### 2.2 Escala lenta: el bucle integrado

In [ ]:
@dataclass
class ParamInt:
    # --- geometría y malla del bucle lento ---
    Lx: float = 10.0            # cm   longitud (fondo -> píloro)
    Ly: float = 4.0             # cm   anchura de la sección
    nx: int = 25
    ny: int = 10
    x_sec: float = 6.0          # cm   las paredes secretan para x < x_sec (cuerpo y fondo)
    frac_ini: float = 0.6       # —    la comida sólida empieza en el 60 % proximal
    # --- química (fase 1b, tampón en equilibrio) ---
    k_a1: float = 1.7e-3; k_a2: float = 0.83; pKa: tuple = (2.5, 3.5, 4.5, 5.5, 6.5)
    beta_prot: float = 0.30     # mmol/(g·pH)
    r_acid: float = 0.375       # mmol/min (estimulada)
    q_s: float = 2.5            # mL/min   caudal de jugo (estimulado)
    r_G: float = 1.0e-4         # mmol/min
    f_basal: float = 0.10
    D_H: float = 5.6e-3         # cm²/min difusión molecular del H+
    escala_D: float = 1.0       # factor sobre la dispersión medida con LBM (incertidumbre del promediado)
    # --- vaciado ---
    kcal_min: float = 2.0
    V_res: float = 30.0
    # --- partículas (fase 4) ---
    rho_p: float = 1.05e-3; d50_ini: float = 3.0; sg_ini: float = 0.5
    q_atrapada: float = 0.5; g_min: float = 0.5; F_min: float = 0.2; F_max: float = 2.0
    sigma_c0: float = 1.0e5; w_inf: float = 0.3; X_w: float = 60.0; pH_ref: float = 4.0
    r_e0: float = 5e-3; pH_opt: float = 2.0; sigma_pH: float = 1.5; K_E: float = 0.005
    d_c: float = 1.5
    gamma_pared: float = 0.5    # —    cuánto arrastra la pared que se contrae a las partículas apoyadas (0: nada, 1: todo)
    n_part: int = 4000; n_max: int = 30000


@dataclass
class Comida6:
    nombre: str
    V_comida: float        # mL (sólidos + bebida)
    prot_tampon: float     # g de proteína que aporta tampón
    masa_solida: float     # g de sólido marcado (partículas)
    kcal: float
    pH0: float = 6.0
    V_basal: float = 25.0


def resolver_h_vec(T, Bt, pKa, h0=None, tol=1e-12, max_iter=60):
    n = len(pKa); Ka = [10 ** (3.0 - pk) for pk in pKa]; Bj = Bt / n
    h = np.clip(T if h0 is None else h0, 1e-12, None); h = np.minimum(h, np.maximum(T, 1e-12))
    for _ in range(max_iter):
        F = h - T; dF = np.ones_like(h)
        for K in Ka:
            F = F + Bj * h / (h + K); dF = dF + Bj * K / (h + K) ** 2
        hn = np.where(h - F / dF <= 0, 0.1 * h, h - F / dF)
        if np.max(np.abs(hn - h) / hn) < tol:
            return hn
        h = hn
    return h


def difusion_variable(c, Dx_caras, Dy_celdas, dx, dy):
    # div(D grad c) con D(x) variable y paredes sin flujo. c: (..., ny, nx)
    Fx = np.zeros(c.shape[:-1] + (c.shape[-1] + 1,))
    Fx[..., 1:-1] = -Dx_caras[1:-1] * np.diff(c, axis=-1) / dx
    Fy = np.zeros(c.shape[:-2] + (c.shape[-2] + 1, c.shape[-1]))
    Fy[..., 1:-1, :] = -Dy_celdas * np.diff(c, axis=-2) / dy
    return -(Fx[..., 1:] - Fx[..., :-1]) / dx - (Fy[..., 1:, :] - Fy[..., :-1, :]) / dy


iT, iB, iG, iE, iP = range(5)
NOMBRES_CAMPOS = ["T", "B", "G", "E", "P"]


class ModeloIntegrado:
    # Bucle lento único: química local + transporte + partículas lagrangianas + píloro abierto + volumen variable.
    # El flujo resuelto con LBM (escala rápida) entra a través de mapas promediados en fase:
    # dispersión del líquido D(x), dispersión y frecuencia de molienda de las partículas.
    # Además, el vaciado genera un flujo neto hacia el píloro u_b(x) (el contenido avanza al salir por el píloro).
    def __init__(self, pi, comida, mapas, semilla=0, guardar_cada=1.0):
        self.p, self.c, self.m = pi, comida, mapas
        p = pi
        self.rng = np.random.default_rng(semilla)
        self.guardar_cada = guardar_cada
        self.V0 = comida.V_comida + comida.V_basal
        self.T_e = comida.kcal / p.kcal_min
        self.dx, self.dy = p.Lx / p.nx, p.Ly / p.ny
        self.x = (np.arange(p.nx) + 0.5) * self.dx
        self.xf = np.arange(p.nx + 1) * self.dx
        self.xm = mapas["x"]
        self.Dx = p.escala_D * np.interp(self.x, self.xm, mapas["fluido"]["Dx"])
        self.Dy = p.escala_D * np.interp(self.x, self.xm, mapas["fluido"]["Dy"])
        self.Dx_caras = np.interp(self.xf, self.x, self.Dx)
        self.Dx_p = p.escala_D * mapas["particula"]["Dx"]; self.lam_g = mapas["particula"]["lambda_g"]
        self.mask_sec = np.zeros((p.ny, p.nx)); self.mask_sec[[0, -1], :] = self.x < p.x_sec
        self.n_sec = self.mask_sec.sum()
        # cantidades por celda (mmol para T, B, G, E; g para P)
        V = self.V0; Vcel = self.Vcel(V)
        n = len(p.pKa)
        Bt = p.beta_prot * comida.prot_tampon * (max(p.pKa) - min(p.pKa) + 1) / V * 1000      # mM
        h0 = 10 ** (3 - comida.pH0)
        ligados = sum((Bt / n) / (1 + 10 ** (comida.pH0 - pk)) for pk in p.pKa)
        H_bas = 10 ** (3 - 1.5) * comida.V_basal / V
        conc = np.zeros((5, p.ny, p.nx))
        conc[iT] = ligados + h0 + H_bas; conc[iB] = Bt; conc[iE] = 0.04 * comida.V_basal / V
        self.N = conc * Vcel / 1000
        self.V = V
        self.h = resolver_h_vec(conc[iT], conc[iB], p.pKa)
        # partículas (lognormal en masa) en la parte proximal
        d = np.exp(self.rng.normal(np.log(p.d50_ini) - 3 * p.sg_ini ** 2, p.sg_ini, p.n_part))
        self.d = d; self.w = np.full(d.size, comida.masa_solida / self.masa(d).sum())
        self.xp = self.rng.uniform(0, p.frac_ini * p.Lx, d.size); self.X = np.zeros(d.size)
        self.sale = {"T": 0.0, "B": 0.0, "G": 0.0, "E": 0.0, "P": 0.0, "solido": 0.0, "liquido": 0.0}
        self.secretado = {"T": 0.0, "G": 0.0}
        self.t = 0.0
        self.hist = {k: [] for k in ("t", "V", "solido_g", "P_g", "pH_medio", "pH_min", "pH_max", "pH_particulas",
                                     "d50", "n_part", "salida_solida_g", "salida_disuelta_g", "u_piloro")}
        self.instantaneas = []

    # ---------- utilidades ----------
    def masa(self, d):
        return self.p.rho_p * np.pi * d ** 3 / 6

    def Vcel(self, V):
        return V / (self.p.nx * self.p.ny)                     # mL por celda (la losa se adelgaza uniformemente)

    def concentraciones(self, N=None, V=None):
        N = self.N if N is None else N
        return N / self.Vcel(self.V if V is None else V) * 1000

    def caudales(self, t):
        p = self.p
        if t < self.T_e:
            return p.q_s, p.q_s + (self.V0 - p.V_res) / self.T_e, 1.0
        return p.f_basal * p.q_s, p.f_basal * p.q_s, p.f_basal

    def velocidad_neta(self, V, q_in, Q):
        # u_b en las caras x (cm/min) a partir del balance de volumen por columnas:
        # lo que entra por la pared (jugo) menos lo que pierde la columna al adelgazarse se va hacia el píloro.
        p = self.p
        W = V / (p.Lx * p.Ly)
        dVdt = q_in - Q
        sec_col = self.mask_sec.sum(axis=0) / max(self.n_sec, 1) * q_in            # mL/min que entran en cada columna
        perdida_col = dVdt / p.nx                                                    # mL/min (uniforme)
        caudal_caras = np.concatenate([[0.0], np.cumsum(sec_col - perdida_col)])     # mL/min a través de cada cara
        return caudal_caras / (p.Ly * W), caudal_caras

    def flujo_transversal(self, V, q_in, Q):
        p = self.p
        _, caudal = self.velocidad_neta(V, q_in, Q)
        sec = self.mask_sec / max(self.n_sec, 1) * q_in                              # mL/min que entran en cada celda
        axial = (caudal[1:] - caudal[:-1]) / p.ny                                    # mL/min que salen de cada fila hacia x+
        encoge = (q_in - Q) / (p.nx * p.ny)                                          # mL/min de cambio de volumen por celda
        neto = sec - axial[None, :] - encoge                                         # (ny, nx): lo que debe salir en vertical
        G = np.zeros((p.ny + 1, p.nx))
        G[1:] = np.cumsum(neto, axis=0)                                              # flujo hacia arriba por cada cara
        G[-1] = 0.0                                                                  # (es cero por construcción)
        return G

    def rhs(self, N, h, V, q_in, Q, f_sec):
        p = self.p
        Vc = self.Vcel(V)
        c = N / Vc * 1000
        dN = difusion_variable(c, self.Dx_caras, self.Dy, self.dx, self.dy) * Vc / 1000
        dN[iT] += difusion_variable(h, np.full(p.nx + 1, p.D_H), np.full(p.nx, p.D_H), self.dx, self.dy) * Vc / 1000
        # advección por el flujo neto hacia el píloro (upwind; u_b >= 0). La última cara es la salida pilórica.
        u, caudal = self.velocidad_neta(V, q_in, Q)
        flujo = np.zeros(c.shape[:-1] + (p.nx + 1,))
        flujo[..., 1:] = (caudal[1:] / p.ny) * c / 1000                            # mmol/min (o g/min) por fila
        dN += flujo[..., :-1] - flujo[..., 1:]
        # flujo transversal: el jugo entra por las filas de pared y empuja el contenido hacia el centro
        # (balance de volumen fila a fila; así el jugo se diluye y nada supera su concentración)
        G = self.flujo_transversal(V, q_in, Q)                                       # (ny+1, nx) mL/min
        cu = np.where(G[1:-1] > 0, c[..., :-1, :], c[..., 1:, :])                    # upwind
        Fy = np.zeros(c.shape[:-2] + (p.ny + 1, p.nx))
        Fy[..., 1:-1, :] = G[1:-1] * cu / 1000
        dN += Fy[..., :-1, :] - Fy[..., 1:, :]
        # química: activación del pepsinógeno
        r_a = (p.k_a1 * h + p.k_a2 * h * c[iE]) * c[iG] * Vc / 1000
        dN[iG] -= r_a; dN[iE] += r_a
        # secreción en las paredes
        dN[iT] += self.mask_sec * f_sec * p.r_acid / self.n_sec
        dN[iG] += self.mask_sec * f_sec * p.r_G / self.n_sec
        salida = flujo[..., -1].sum(axis=-1)                                        # por especie
        return dN, salida

    def paso_campos(self, dt):
        p = self.p
        q_in, Q, f_sec = self.caudales(self.t)
        V0 = self.V; V1 = V0 + (q_in - Q) * dt
        N0, h0 = self.N, self.h
        k1, s1 = self.rhs(N0, h0, V0, q_in, Q, f_sec)
        N1 = N0 + dt * k1
        c1 = N1 / self.Vcel(V1) * 1000
        h1 = resolver_h_vec(c1[iT], c1[iB], p.pKa, h0)
        k2, s2 = self.rhs(N1, h1, V1, q_in, Q, f_sec)
        self.N = N0 + 0.5 * dt * (k1 + k2)
        sal = 0.5 * dt * (s1 + s2)
        for k, i in zip(NOMBRES_CAMPOS, range(5)):
            self.sale[k] += float(sal[i])
        self.secretado["T"] += f_sec * p.r_acid * dt; self.secretado["G"] += f_sec * p.r_G * dt
        self.sale["liquido"] += Q * dt
        self.V = V1
        c = self.concentraciones()
        self.h = resolver_h_vec(c[iT], c[iB], p.pKa, h1)
        return q_in, Q

    def paso(self, dt):
        Dmax = max(self.Dx.max(), self.Dy.max()) + self.p.D_H
        dt_dif = 0.35 / (Dmax * (1 / self.dx ** 2 + 1 / self.dy ** 2))
        q_in, Q, _ = self.caudales(self.t)
        u, _ = self.velocidad_neta(self.V, q_in, Q)
        dt_adv = 0.5 * self.dx / max(u.max(), 1e-9)
        G = self.flujo_transversal(self.V, q_in, Q)
        dt_adv = min(dt_adv, 0.5 * self.Vcel(self.V) / max(np.abs(G).max(), 1e-9))
        n_sub = int(math.ceil(dt / min(dt_dif, dt_adv)))
        for _ in range(n_sub):
            q_in, Q = self.paso_campos(dt / n_sub)
        self.paso_particulas(dt, q_in, Q)
        self.t += dt

    def paso_particulas(self, dt, q_in, Q):
        p, rng = self.p, self.rng
        if self.d.size == 0:
            return
        # transporte: flujo neto hacia el píloro + dispersión axial medida con LBM (Itô, D variable)
        u, _ = self.velocidad_neta(self.V, q_in, Q)
        # las partículas sedimentadas avanzan con la velocidad del líquido a la altura de su centro (perfil parabólico)
        yc = np.minimum(0.5 * self.d / 10, 0.5 * p.Ly)                             # cm
        alfa = 6 * (yc / p.Ly) * (1 - yc / p.Ly)
        ub = alfa * np.interp(self.xp, self.xf, u)
        # además, apoyadas en la pared, se mueven con ella: el estómago se contrae (tono) al vaciarse,
        # con L ∝ V^(1/3), y el píloro es el punto fijo -> deriva hacia el píloro proporcional a la distancia
        dlnL = (q_in - Q) / (3 * self.V)
        ub = ub - p.gamma_pared * (p.Lx - self.xp) * dlnL
        D = np.interp(self.xp, self.xm, self.Dx_p); dD = np.interp(self.xp, self.xm, np.gradient(self.Dx_p, self.xm))
        xn = self.xp + (ub + dD) * dt + np.sqrt(2 * D * dt) * rng.standard_normal(self.xp.size)
        xn = np.abs(xn)                                                     # fondo: reflexión
        cruza = xn >= p.Lx
        pasa = cruza & (self.d < p.d_c)                                     # criba pilórica
        self.sale["solido"] += float(np.sum(self.w[pasa] * self.masa(self.d[pasa])))
        xn = np.where(cruza & ~pasa, 2 * p.Lx - xn - 1e-9, xn)              # las grandes vuelven (retropulsión)
        quedan = ~pasa
        for nombre, val in (("xp", xn), ("d", self.d), ("w", self.w), ("X", self.X)):
            setattr(self, nombre, val[quedan])
        if self.d.size == 0:
            return
        i = np.minimum((self.xp / self.dx).astype(int), p.nx - 1)
        c = self.concentraciones()
        pH = 3 - np.log10(self.h[0, i]); E = c[iE][0, i]                   # viven en la capa del fondo
        # ablandamiento por el ácido LOCAL y rotura en la zona de molienda
        self.X += np.maximum(0.0, p.pH_ref - pH) * dt
        sig = p.sigma_c0 * (p.w_inf + (1 - p.w_inf) * np.exp(-self.X / p.X_w))
        Prot = np.clip((p.F_max - sig * (self.d * 1e-3) ** 2) / (p.F_max - p.F_min), 0, 1)
        # en cada contracción terminal (frecuencia λ_g) la partícula atrapada se rompe si es grande
        # o, si cabe por el píloro, sale expulsada con el pulso transpilórico (criba antral)
        lam = np.interp(self.xp, self.xm, self.lam_g) * p.q_atrapada
        expulsa = (self.d < p.d_c) & (rng.random(self.d.size) < 1 - np.exp(-lam * dt))
        self.sale["solido"] += float(np.sum(self.w[expulsa] * self.masa(self.d[expulsa])))
        for nombre in ("xp", "d", "w", "X"):
            setattr(self, nombre, getattr(self, nombre)[~expulsa])
        i, pH, E, sig = i[~expulsa], pH[~expulsa], E[~expulsa], sig[~expulsa]
        Prot = np.clip((p.F_max - sig * (self.d * 1e-3) ** 2) / (p.F_max - p.F_min), 0, 1)
        S = np.interp(self.xp, self.xm, self.lam_g) * p.q_atrapada * (self.d > p.g_min) * Prot
        rompe = rng.random(self.d.size) < 1 - np.exp(-S * dt)
        fac = 2 ** (-1 / 3)
        self.d = np.concatenate([np.where(rompe, self.d * fac, self.d), self.d[rompe] * fac])
        for nombre in ("w", "xp", "X"):
            setattr(self, nombre, np.concatenate([getattr(self, nombre), getattr(self, nombre)[rompe]]))
        i = np.concatenate([i, i[rompe]]); pH = np.concatenate([pH, pH[rompe]]); E = np.concatenate([E, E[rompe]])
        # erosión con pH y pepsina LOCALES; lo erosionado se convierte en péptidos disueltos en esa celda
        r = p.r_e0 * np.exp(-((pH - p.pH_opt) / p.sigma_pH) ** 2) * E / (E + p.K_E)
        d_n = np.maximum(self.d - r * dt, 0.0)
        dm = self.w * (self.masa(self.d) - self.masa(d_n))
        np.add.at(self.N[iP][0], i, dm)
        vivo = d_n > 0
        self.d = d_n
        for nombre in ("d", "w", "xp", "X"):
            setattr(self, nombre, getattr(self, nombre)[vivo])
        if self.d.size > p.n_max:                                             # aclarado estratificado (fase 4)
            clase = np.floor(3 * np.log2(self.d)).astype(int)
            _, inv, cuenta = np.unique(clase, return_inverse=True, return_counts=True)
            poblada = cuenta[inv] > 200
            quita = poblada & (rng.random(self.d.size) < 0.5)
            self.w = np.where(poblada, 2 * self.w, self.w)
            for nombre in ("d", "w", "xp", "X"):
                setattr(self, nombre, getattr(self, nombre)[~quita])

    # ---------- registro ----------
    def totales(self):
        tot = {k: float(self.N[i].sum()) for k, i in zip(NOMBRES_CAMPOS, range(5))}
        tot["solido"] = float(np.sum(self.w * self.masa(self.d)))
        return tot

    def registrar(self, instantanea=False):
        H = self.hist; tot = self.totales()
        pHc = 3 - np.log10(self.h)
        H["t"].append(self.t); H["V"].append(self.V)
        H["solido_g"].append(tot["solido"]); H["P_g"].append(tot["P"])
        H["pH_medio"].append(float(3 - np.log10(np.mean(self.h))))
        H["pH_min"].append(float(pHc.min())); H["pH_max"].append(float(pHc.max()))
        if self.d.size:
            i = np.minimum((self.xp / self.dx).astype(int), self.p.nx - 1)
            ms = self.w * self.masa(self.d); o = np.argsort(self.d)
            H["pH_particulas"].append(float(np.sum(ms * pHc[0, i]) / ms.sum()))
            H["d50"].append(float(np.interp(0.5, np.cumsum(ms[o]) / ms.sum(), self.d[o])))
        else:
            H["pH_particulas"].append(np.nan); H["d50"].append(np.nan)
        H["n_part"].append(int(self.d.size))
        H["salida_solida_g"].append(self.sale["solido"]); H["salida_disuelta_g"].append(self.sale["P"])
        q_in, Q, _ = self.caudales(self.t)
        H["u_piloro"].append(float(self.velocidad_neta(self.V, q_in, Q)[0][-1]))
        if instantanea:
            self.instantaneas.append({"t": self.t, "pH": pHc.copy(), "P": self.concentraciones()[iP].copy(), "V": self.V,
                                      "xp": self.xp.copy(), "d": self.d.copy(), "w": self.w.copy()})

    def simular(self, t_final, dt, cada_inst=None, informar=True):
        n = int(round(t_final / dt)); cada = max(1, int(round(self.guardar_cada / dt)))
        cada_i = None if cada_inst is None else max(1, int(round(cada_inst / dt)))
        t0 = time.time()
        for k in range(n + 1):
            if k % cada == 0:
                self.registrar(instantanea=(cada_i is not None and k % cada_i == 0))
            if k == n:
                break
            self.paso(dt)
            if informar and (k + 1) % max(1, n // 4) == 0:
                print(f"  t = {self.t:5.0f} min  ({time.time() - t0:4.0f} s)")
        self.hist = {k: np.array(v) for k, v in self.hist.items()}
        return self

    def retencion(self):
        H = self.hist
        return 100 * (H["solido_g"] + H["P_g"]) / self.c.masa_solida

## 3. Escala rápida: del flujo LBM a los mapas

Cargamos el campo de una onda (`campo_antro.npz` de las fases 3–5) o lo regeneramos. Después seguimos 1500 trazadores de fluido y 1500 partículas de 3 mm durante 6 ondas.

In [ ]:
CLAVES = ("t", "ux", "uy", "solido", "x", "y", "dx", "periodo", "nu")
datos = dict(np.load("campo_antro.npz")) if os.path.exists("campo_antro.npz") else {}
if any(c not in datos for c in CLAVES):
    print("Regenerando el flujo con LBM (≈ 75 s)...")
    exec(CODIGO_LBM, globals())
    pa = ParamAntro(); R = Red(np)
    S = simular_antro(pa, R, periodos=2, n_particulas=10, informar=False)
    t = S["t"]; k = int(round(pa.T / (t[1] - t[0]))); ult = slice(len(t) - k - 1, len(t)); dx = S["U"].dx
    datos = dict(t=t[ult] - t[ult][0], ux=np.nan_to_num(S["ux"][ult]), uy=np.nan_to_num(S["uy"][ult]), solido=S["solido"][ult],
                 x=(np.arange(S["nx"]) - 0.5) * dx, y=(np.arange(S["ny"]) + 0.5) * dx, dx=dx, periodo=pa.T, nu=pa.nu,
                 tau_max=np.nan_to_num(S["tau_max"][ult]))
    np.savez_compressed("campo_antro.npz", **datos)
CAMPO = CampoPeriodico(datos)
t0 = time.time()
MAPAS = mapas_escala_rapida(CAMPO, ondas=6, n=1500, ventana=2)
print(f"Mapas de la escala rápida: {time.time() - t0:.0f} s")
display(pd.DataFrame({"x [cm]": MAPAS["x"], "D líquido axial [cm²/min]": MAPAS["fluido"]["Dx"],
                      "D líquido transversal [cm²/min]": MAPAS["fluido"]["Dy"],
                      "D partículas axial [cm²/min]": MAPAS["particula"]["Dx"],
                      "λg molienda [1/min]": MAPAS["particula"]["lambda_g"]}).style.format(precision=3).hide(axis="index"))

### V1 — ¿Reproduce el promediado la dispersión del LBM?
Partiendo de las mismas posiciones, comparamos el desplazamiento cuadrático medio de los trazadores en el flujo LBM real con el de un **paseo aleatorio** que usa solo el mapa $D(x)$. El paseo es de Itô con $D$ variable: $dx = D'(x)\,dt + \sqrt{2D\,dt}\;\xi$.

In [ ]:
X = MAPAS["fluido"]["x_tray"]; idx = np.arange(0, X.shape[0], int(round(CAMPO.T / 0.5)))
x0 = X[idx[0]]
rw = paseo_aleatorio(x0, MAPAS["fluido"]["Dx"], MAPAS["x"], CAMPO.T / 60, len(idx) - 1)
ondas = np.arange(len(idx))
msd_lbm = np.array([np.mean((X[i] - x0) ** 2) for i in idx]); msd_rw = np.mean((rw - x0) ** 2, axis=1)
# El paseo dispersa menos que el LBM (transporte coherente de los chorros). Calibramos un único factor
# con la PROPIA escala rápida (no con datos clínicos): que el paseo reproduzca el LBM a 4 ondas.
ESCALA_D = float(msd_lbm[4] / msd_rw[4])
rw_cal = paseo_aleatorio(x0, ESCALA_D * MAPAS["fluido"]["Dx"], MAPAS["x"], CAMPO.T / 60, len(idx) - 1, semilla=1)
msd_cal = np.mean((rw_cal - x0) ** 2, axis=1)
cociente = msd_cal[6] / msd_lbm[6]
print(f"Factor de calibración del promediado: escala_D = {ESCALA_D:.2f}; comprobación a 6 ondas (no usada al calibrar): {cociente:.2f}")

fig, axs = plt.subplots(1, 3, figsize=(15, 3.9))
a = axs[0]
s = CAMPO.sol[len(CAMPO.t) // 3]
sp = np.hypot(CAMPO.ux[len(CAMPO.t) // 3], CAMPO.uy[len(CAMPO.t) // 3])
a.imshow(np.where(s, np.nan, sp), origin="lower", extent=[CAMPO.x[0], CAMPO.x[-1], CAMPO.y[0] - 1, CAMPO.y[-1] - 1], cmap="magma", aspect="equal")
a.set(title="Escala rápida: rapidez en una fase de la onda", xlabel="x [cm]", ylabel="y [cm]"); a.grid(False)
a = axs[1]
a.step(MAPAS["bordes"], np.r_[MAPAS["fluido"]["Dx"], MAPAS["fluido"]["Dx"][-1]], where="post", color=C1, label="D líquido axial")
a.step(MAPAS["bordes"], np.r_[MAPAS["fluido"]["Dy"], MAPAS["fluido"]["Dy"][-1]], where="post", color=C3, label="D líquido transversal")
a2 = a.twinx(); a2.step(MAPAS["bordes"], np.r_[MAPAS["particula"]["lambda_g"], MAPAS["particula"]["lambda_g"][-1]], where="post", color=C2, ls="--")
a2.set_ylabel("λg: contracciones que atrapan / min", color=C2); a2.grid(False)
a.set(xlabel="x: fondo → píloro [cm]", ylabel="D [cm²/min]", title="Mapas promediados en fase"); a.legend(loc="upper left", fontsize=9)
a = axs[2]
a.plot(ondas, msd_lbm, "o-", color=C5, label="LBM directo")
a.plot(ondas, msd_rw, "s--", color=C4, label="paseo con D(x) medido")
a.plot(ondas, msd_cal, "^-", color=C2, label=f"paseo con {ESCALA_D:.2f}·D(x)")
a.axvline(4, color=GRIS_MEDIO, lw=0.8, ls=":"); a.text(4.1, msd_lbm.max() * 0.05, "calibración", fontsize=8.5, color=TINTA2)
a.set(xlabel="ondas", ylabel="⟨Δx²⟩ [cm²]", title=f"V1 · Promediado: a 6 ondas paseo/LBM = {cociente:.2f}"); a.legend(fontsize=9)
fig.tight_layout(); fig.savefig("figuras/01_escala_rapida.png", dpi=200, bbox_inches="tight"); plt.show()
VERIFICACION.append({"test": "V1 promediado en fase: dispersión (paseo calibrado/LBM, 6 ondas)", "valor": cociente,
                     "criterio": "entre 0.5 y 2", "pasa": 0.5 < cociente < 2})

**Lectura.** En el LBM la dispersión crece más deprisa que en un paseo aleatorio con el $D(x)$ medido en ventanas de 2 ondas. Hay transporte coherente, como los chorros, que no es puramente difusivo.

Para corregirlo calibramos **un solo factor** (`escala_D`) con la propia escala rápida, sin usar datos clínicos, de forma que el paseo reproduzca el LBM a 4 ondas. Luego comprobamos a 6 ondas, un dato que no se usó al calibrar. La incertidumbre restante del promediado se recorre en el conjunto de la sección 5, entre 0.5 y 2 veces ese factor.

Dos datos del mapa marcan toda la dinámica:
- En el fondo y el cuerpo ($x < 4$ cm) no hay ondas y el líquido apenas se mezcla.
- Las partículas que sedimentan apenas se dispersan por sí solas. Las mueve el **flujo neto hacia el píloro** y la contracción de la pared al vaciarse el estómago.

## 4. Simulación integrada: comida de consenso y comida de 1000 kcal

In [ ]:
CONSENSO = Comida6("consenso", V_comida=315, prot_tampon=17, masa_solida=13, kcal=255)
MIL = Comida6("1000 kcal", V_comida=650, prot_tampon=40, masa_solida=40, kcal=1000)
PI = ParamInt(escala_D=ESCALA_D)
DT = 0.02                                   # min (≈ 1.2 s); los campos se subdividen si la estabilidad lo pide
t0 = time.time()
M6 = ModeloIntegrado(PI, CONSENSO, MAPAS, semilla=0, guardar_cada=1.0)
T0_protones = M6.totales()["T"]
M6.simular(240, DT, cada_inst=3.0)
print(f"Comida de consenso: {time.time() - t0:.0f} s")
t0 = time.time()
M6k = ModeloIntegrado(PI, MIL, MAPAS, semilla=0).simular(300, DT, informar=False)
print(f"Comida de 1000 kcal: {time.time() - t0:.0f} s")

def metricas6(mod_c, mod_k):
    H, Hk = mod_c.hist, mod_k.hist
    ret = mod_c.retencion()
    bajo2 = np.where(Hk["pH_medio"] <= 2.0)[0]
    return {"retención 1 h [%]": float(np.interp(60, H["t"], ret)), "retención 2 h [%]": float(np.interp(120, H["t"], ret)),
            "retención 4 h [%]": float(np.interp(240, H["t"], ret)), "pH máximo": float(Hk["pH_medio"].max()),
            "t hasta pH ≤ 2 [min]": float(Hk["t"][bajo2[0]]) if bajo2.size else np.nan}
MET6 = metricas6(M6, M6k)
display(pd.DataFrame([MET6]).style.format(precision=2).hide(axis="index"))

### Verificación del bucle integrado
- **V2 masa de comida**: sólido + péptidos disueltos + lo vaciado (sólido y disuelto) = masa ingerida.
- **V3 protones**: lo que hay + lo vaciado − lo secretado = lo inicial.
- **V4 límite físico del pH**: ninguna celda puede ser más ácida que el propio jugo gástrico. Esto comprueba el flujo transversal que diluye el jugo al entrar.
- **V5 discretización**: repetimos con la mitad de paso de tiempo y una malla más fina (40 × 16). La diferencia incluye también el ruido estadístico de las partículas.

In [ ]:
tot = M6.totales()
e_masa = abs(tot["solido"] + tot["P"] + M6.sale["solido"] + M6.sale["P"] - CONSENSO.masa_solida) / CONSENSO.masa_solida
e_prot = abs(tot["T"] + M6.sale["T"] - M6.secretado["T"] - T0_protones) / M6.secretado["T"]
pH_jugo = 3 - math.log10(PI.r_acid / PI.q_s * 1000)
pH_min = float(np.min(M6.hist["pH_min"]))
t0 = time.time()
M6f = ModeloIntegrado(replace(PI, nx=40, ny=16), CONSENSO, MAPAS, semilla=1).simular(240, DT / 2, informar=False)
M6kf = ModeloIntegrado(replace(PI, nx=40, ny=16), MIL, MAPAS, semilla=1).simular(300, DT / 2, informar=False)
MET6f = metricas6(M6f, M6kf)
ENUM6 = {k: abs(MET6[k] - MET6f[k]) for k in MET6}
print(f"Malla fina y Δt/2: {time.time() - t0:.0f} s")
display(pd.DataFrame([MET6, MET6f, ENUM6], index=["nominal 25×10, Δt", "fina 40×16, Δt/2", "diferencia"]).style.format(precision=2))
VERIFICACION += [
    {"test": "V2 conservación de la masa de comida", "valor": e_masa, "criterio": "< 1e-10", "pasa": e_masa < 1e-10},
    {"test": "V3 conservación de protones", "valor": e_prot, "criterio": "< 1e-10", "pasa": e_prot < 1e-10},
    {"test": "V4 pH nunca más ácido que el jugo", "valor": pH_min - pH_jugo, "criterio": "≥ −0.01", "pasa": pH_min - pH_jugo > -0.01},
    {"test": "V5 discretización: retenciones (puntos %)", "valor": max(ENUM6[k] for k in ENUM6 if "retención" in k),
     "criterio": "< 8 puntos", "pasa": max(ENUM6[k] for k in ENUM6 if "retención" in k) < 8},
]

## 5. Representación: cuatro horas de digestión

### 5.1 Instantáneas: pH local, péptidos disueltos y partículas
Cada columna es un instante. Fila superior: pH, con las partículas encima; su tamaño y su color indican el diámetro y la línea blanca marca el corte del píloro. Fila inferior: péptidos disueltos, producto de la erosión. El píloro está a la derecha.

In [ ]:
instantes = [0, 30, 60, 90, 120, 180]
Ly = PI.Ly
fig, axs = plt.subplots(2, len(instantes), figsize=(17, 3.6), gridspec_kw={"hspace": 0.05})
cmap_d = plt.get_cmap("plasma")
for j, ti in enumerate(instantes):
    s = min(M6.instantaneas, key=lambda z: abs(z["t"] - ti))
    W = s["V"] / (PI.Lx * PI.Ly)
    for fila, (campo, cm, vmin, vmax, et) in enumerate([(s["pH"], "viridis", 0.8, 6.0, "pH"),
                                                        (s["P"], "Oranges", 0, max(1e-6, max(np.max(z["P"]) for z in M6.instantaneas)), "péptidos [g/L]")]):
        a = axs[fila, j]
        im = a.imshow(campo, origin="lower", extent=[0, PI.Lx, 0, Ly], cmap=cm, vmin=vmin, vmax=vmax, aspect="equal")
        a.axvline(PI.Lx, color="k", lw=3)
        if fila == 0 and s["d"].size:
            rng_ = np.random.default_rng(j)
            m = rng_.choice(s["d"].size, min(400, s["d"].size), replace=False)
            a.scatter(s["xp"][m], 0.1 + 0.25 * rng_.random(m.size), s=4 + 6 * s["d"][m], c=s["d"][m], cmap=cmap_d,
                      vmin=0, vmax=5, edgecolors="white", linewidths=0.3, zorder=3)
            a.set_title(f"t = {ti} min · V = {s['V']:.0f} mL", fontsize=10)
        a.set_xticks([0, 5, 10] if fila == 1 else []); a.set_yticks([]); a.grid(False)
        if j == 0:
            a.set_ylabel(et)
        if j == len(instantes) - 1:
            plt.colorbar(im, ax=axs[fila, :], shrink=0.85, pad=0.01, label=et)
axs[1, 0].set_xlabel("x [cm] (píloro →)")
sm = plt.cm.ScalarMappable(cmap=cmap_d, norm=plt.Normalize(0, 5))
fig.suptitle("Modelo integrado: comida de consenso (partículas: diámetro en color y tamaño, 0–5 mm)", fontweight="bold")
fig.savefig("figuras/02_instantaneas.png", dpi=200, bbox_inches="tight"); plt.show()

**Lectura.** Desde el primer momento aparece una **capa ácida** junto a las paredes secretoras del cuerpo. La parte distal y el centro del contenido siguen tamponados durante decenas de minutos: es la bolsa ácida de la fase 2, ahora con vaciado. Las partículas, sedimentadas en la pared inferior, avanzan hacia el antro conforme el estómago se vacía y se contrae. Allí las trituran las contracciones terminales y las de menos de 1.5 mm salen por el píloro. La erosión deja un rastro de péptidos disueltos que el flujo neto lleva también hacia el píloro.

### 5.2 Incertidumbre paramétrica (conjunto pequeño)
Cada simulación integrada tarda unos 20–30 s, así que el conjunto es necesariamente pequeño (`N_ENS`). La banda es **indicativa**: con 6 miembros, los percentiles 5–95 son poco precisos.

In [ ]:
N_ENS = 6                 # súbelo si tienes tiempo (≈ 25 s por miembro); 0 para saltar
N_ENS_K = 3               # para la comida de 1000 kcal
RANGOS6 = [("kcal_min", 1.0, 4.0, "log"), ("q_atrapada", 0.2, 0.8, "lin"), ("sigma_c0", 3e4, 3e5, "log"),
           ("escala_D", 0.5 * ESCALA_D, 2.0 * ESCALA_D, "log"), ("d50_ini", 2.0, 4.0, "lin"), ("r_e0", 1e-3, 1e-2, "log"),
           ("gamma_pared", 0.0, 1.0, "lin"), ("d_c", 1.0, 2.0, "lin"), ("beta_prot", 0.15, 0.45, "lin"), ("r_acid", 0.19, 0.75, "log")]
def muestras6(n, semilla):
    u = qmc.LatinHypercube(d=len(RANGOS6), seed=semilla).random(n)
    return [{nom: (lo * (hi / lo) ** x if esc == "log" else lo + (hi - lo) * x) for (nom, lo, hi, esc), x in zip(RANGOS6, f)} for f in u]
t0 = time.time()
ENS_C, ENS_K = [], []
for k, mu in enumerate(muestras6(N_ENS, 5)):
    ENS_C.append(ModeloIntegrado(replace(PI, **mu), CONSENSO, MAPAS, semilla=10 + k).simular(240, DT, informar=False))
for k, mu in enumerate(muestras6(N_ENS_K, 6)):
    ENS_K.append(ModeloIntegrado(replace(PI, **mu), MIL, MAPAS, semilla=20 + k).simular(300, DT, informar=False))
print(f"Conjunto ({N_ENS} + {N_ENS_K} simulaciones): {time.time() - t0:.0f} s")
filas_c = []
for mc in ENS_C:
    r = mc.retencion(); H = mc.hist
    filas_c.append({m: float(np.interp(tt, H["t"], r)) for m, tt in (("retención 1 h [%]", 60), ("retención 2 h [%]", 120), ("retención 4 h [%]", 240))})
filas_k = []
for mk in ENS_K:
    Hk = mk.hist; b = np.where(Hk["pH_medio"] <= 2)[0]
    filas_k.append({"pH máximo": float(Hk["pH_medio"].max()), "t hasta pH ≤ 2 [min]": float(Hk["t"][b[0]]) if b.size else np.nan})
BANDA6 = pd.concat([pd.DataFrame(filas_c), pd.DataFrame(filas_k)], axis=1)
display(BANDA6.describe().loc[["min", "50%", "max"]].style.format(precision=2))

### 5.3 Series temporales frente a los datos

In [ ]:
# Valores de las versiones anteriores (registro de validación si existe; si no, las ejecuciones de las fases 1b y 4)
if os.path.exists("registro_validacion.csv"):
    REG_PREV = pd.read_csv("registro_validacion.csv")
    REG_PREV["versión"] = REG_PREV["versión"].str.split(" ·").str[0].str.strip()
else:
    REG_PREV = pd.DataFrame([
        ("1a", "retención 1 h [%]", 32.08, 6.28, 49.22), ("1a", "retención 2 h [%]", 1.90, 0.47, 18.78), ("1a", "retención 4 h [%]", 0.28, 0.04, 1.06),
        ("1a", "pH máximo", 5.06, 4.56, 5.32), ("1a", "t hasta pH ≤ 2 [min]", 103.35, 42.47, 151.43),
        ("1b", "retención 1 h [%]", 32.08, 6.28, 49.22), ("1b", "retención 2 h [%]", 1.90, 0.47, 18.78), ("1b", "retención 4 h [%]", 0.28, 0.04, 1.06),
        ("1b", "pH máximo", 5.93, 5.49, 6.35), ("1b", "t hasta pH ≤ 2 [min]", 103.35, 42.47, 151.43),
        ("4", "retención 1 h [%]", 72.89, 38.86, 91.93), ("4", "retención 2 h [%]", 24.20, 10.00, 70.59), ("4", "retención 4 h [%]", 7.68, 3.73, 30.99),
    ], columns=["versión", "métrica", "nominal", "p5", "p95"])
OBJ = {"retención 1 h [%]": (37, 90), "retención 2 h [%]": (30, 60), "retención 4 h [%]": (0, 10),
       "pH máximo": (5.8, 6.7), "t hasta pH ≤ 2 [min]": (0, 240)}

fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
a = axs[0]
t6 = M6.hist["t"]
if len(ENS_C):
    curvas = np.array([np.interp(t6, mc.hist["t"], mc.retencion()) for mc in ENS_C])
    a.fill_between(t6, curvas.min(0), curvas.max(0), color=COL_V["6"], alpha=0.15, lw=0, label="6: rango del conjunto")
a.plot(t6, M6.retencion(), color=COL_V["6"], lw=2.4, label="6: integrado (nominal)")
for (lo, hi), tc in zip([OBJ["retención 1 h [%]"], OBJ["retención 2 h [%]"], OBJ["retención 4 h [%]"]], (60, 120, 240)):
    a.plot([tc, tc], [lo, hi], color=GRIS, lw=7, alpha=0.3, solid_capstyle="butt")
for v, mk in (("1b", "s"), ("4", "D")):
    sub = REG_PREV[(REG_PREV["versión"] == v) & REG_PREV["métrica"].str.startswith("retención")]
    tt = [60 if "1 h" in m else 120 if "2 h" in m else 240 for m in sub["métrica"]]
    a.plot(np.array(tt) + (-4 if v == "1b" else 4), sub["nominal"], mk, color=COL_V[v], ms=7, label=f"{v} (nominal)", ls="none")
a.plot([], [], color=GRIS, lw=7, alpha=0.3, label="rango normal medido")
a.set(xlabel="t [min]", ylabel="retención [%]", title="Vaciado de la comida de consenso", ylim=(0, 105)); a.legend(fontsize=8.5)

a = axs[1]
H = M6.hist
a.fill_between(H["t"], H["pH_min"], H["pH_max"], color=C3, alpha=0.15, lw=0, label="rango espacial (mín–máx)")
a.plot(H["t"], H["pH_medio"], color=C3, label="pH medio del contenido")
a.plot(H["t"], H["pH_particulas"], color=C2, lw=1.6, ls="--", label="pH junto a las partículas")
a.axhline(pH_jugo, color=GRIS_MEDIO, lw=0.8, ls=":"); a.text(235, pH_jugo + 0.1, "pH del jugo", ha="right", fontsize=8.5, color=TINTA2)
a.set(xlabel="t [min]", ylabel="pH", title="Acidez: la comida de consenso"); a.legend(fontsize=8.5)

a = axs[2]
m0 = CONSENSO.masa_solida
a.stackplot(H["t"], 100 * H["solido_g"] / m0, 100 * H["P_g"] / m0, 100 * H["salida_solida_g"] / m0, 100 * H["salida_disuelta_g"] / m0,
            colors=[C2, C4, C1, C5], alpha=0.8, labels=["sólido en el estómago", "disuelto en el estómago", "salió como partículas", "salió disuelto"])
a2 = a.twinx(); a2.plot(H["t"], H["d50"], color=GRIS, lw=1.4); a2.set_ylabel("d50 en masa [mm] (línea)", color=GRIS); a2.grid(False)
a.set(xlabel="t [min]", ylabel="% de la comida", title="¿A dónde va la comida?", ylim=(0, 100)); a.legend(fontsize=8, loc="center right")
fig.tight_layout(); fig.savefig("figuras/03_series.png", dpi=200, bbox_inches="tight"); plt.show()

### 5.4 Animación de la digestión completa
pH del contenido, partículas y curva de retención avanzando a la vez, a 3 minutos de simulación por fotograma. Se guarda en `figuras/` como MP4 y GIF.

In [ ]:
ANIMAR = True
if ANIMAR:
    fig = plt.figure(figsize=(12, 4.6))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.6, 1])
    a1 = fig.add_subplot(gs[0]); a2 = fig.add_subplot(gs[1])
    s0 = M6.instantaneas[0]
    im = a1.imshow(s0["pH"], origin="lower", extent=[0, PI.Lx, 0, PI.Ly], cmap="viridis", vmin=0.8, vmax=6, aspect="auto")
    plt.colorbar(im, ax=a1, shrink=0.85, label="pH"); a1.axvline(PI.Lx, color="k", lw=4)
    a1.text(PI.Lx - 0.1, PI.Ly - 0.25, "píloro", ha="right", color="white", fontsize=9)
    sc = a1.scatter([], [], s=[], c=[], cmap="plasma", vmin=0, vmax=5, edgecolors="white", linewidths=0.3, zorder=3)
    a1.set(xlabel="x [cm]", yticks=[]); a1.grid(False); tit = a1.set_title("")
    ret6 = M6.retencion()
    for (lo, hi), tc in zip([OBJ["retención 1 h [%]"], OBJ["retención 2 h [%]"], OBJ["retención 4 h [%]"]], (60, 120, 240)):
        a2.plot([tc, tc], [lo, hi], color=GRIS, lw=7, alpha=0.3, solid_capstyle="butt")
    linea, = a2.plot([], [], color=COL_V["6"], lw=2.4); punto, = a2.plot([], [], "o", color=COL_V["6"])
    a2.set(xlim=(0, 245), ylim=(0, 105), xlabel="t [min]", ylabel="retención [%]", title="Vaciado frente a los datos (gris)")
    rng_a = np.random.default_rng(0)
    def frame(n):
        s = M6.instantaneas[n]
        im.set_data(s["pH"])
        if s["d"].size:
            m = rng_a.choice(s["d"].size, min(400, s["d"].size), replace=False)
            sc.set_offsets(np.column_stack([s["xp"][m], 0.1 + 0.25 * rng_a.random(m.size)]))
            sc.set_sizes(4 + 6 * s["d"][m]); sc.set_array(s["d"][m])
        else:
            sc.set_offsets(np.empty((0, 2)))
        k = int(np.searchsorted(M6.hist["t"], s["t"]))
        linea.set_data(M6.hist["t"][:k + 1], ret6[:k + 1]); punto.set_data([M6.hist["t"][k]], [ret6[k]])
        tit.set_text(f"t = {s['t']:.0f} min · V = {s['V']:.0f} mL · {s['d'].size} partículas numéricas")
        return im, sc, linea, punto, tit
    ANIM = animation.FuncAnimation(fig, frame, frames=range(len(M6.instantaneas)), interval=110, blit=True)
    plt.close(fig)
    display(HTML(ANIM.to_jshtml()))
    for ext, escritor in (("mp4", animation.FFMpegWriter(fps=10, bitrate=1800)), ("gif", animation.PillowWriter(fps=10))):
        try:
            ANIM.save(f"figuras/04_digestion.{ext}", writer=escritor, dpi=100); print(f"figuras/04_digestion.{ext}")
        except Exception as e:
            print(f"{ext}: no disponible ({type(e).__name__})")

## 6. Tabla comparativa: ¿mejora el modelo al integrar?

Las mismas cinco métricas clínicas para todas las versiones:
- retención gástrica a 1, 2 y 4 h de la comida de consenso (gammagrafía);
- pH máximo y tiempo hasta pH ≤ 2 tras una comida de 1000 kcal.

Para cada versión se da el valor nominal, si cae dentro del rango medido (✅/❌) y el **error de modelo normalizado**: la distancia al rango, dividida por su anchura. La versión 4 no tiene métricas de pH porque usaba una curva de pH impuesta.

In [ ]:
def disc(x, lo, hi):
    return 0.0 if lo <= x <= hi else min(abs(x - lo), abs(x - hi))

filas = []
nom6 = {**MET6}
b6 = {m: (np.nanpercentile(BANDA6[m], 5), np.nanpercentile(BANDA6[m], 95)) if (len(BANDA6) and m in BANDA6) else (np.nan, np.nan) for m in OBJ}
for v in ["1a", "1b", "4", "6"]:
    for met, (lo, hi) in OBJ.items():
        if v == "6":
            x = nom6[met]; p5, p95 = b6[met]; e_num = ENUM6[met]
        else:
            sub = REG_PREV[(REG_PREV["versión"] == v) & (REG_PREV["métrica"] == met)]
            if sub.empty:
                filas.append({"versión": v, "métrica": met, "nominal": np.nan}); continue
            x, p5, p95 = sub["nominal"].iloc[0], sub["p5"].iloc[0], sub["p95"].iloc[0]
            e_num = sub["error_numérico"].iloc[0] if "error_numérico" in sub else np.nan
        filas.append({"versión": v, "métrica": met, "nominal": x, "p5": p5, "p95": p95, "dentro": float(lo <= x <= hi),
                      "error_modelo_norm": disc(x, lo, hi) / (hi - lo), "error_numérico": e_num})
COMP = pd.DataFrame(filas)

FISICA = {
    "química (ácido, pepsina)":              ["✓", "✓", "✓", "✓"],
    "vaciado del líquido":                    ["✓", "✓", "✓", "✓"],
    "tampón en equilibrio":                   ["—", "✓", "—", "✓"],
    "partículas + criba pilórica":            ["—", "—", "✓", "✓"],
    "pH y pepsina locales (espacio 2D)":      ["—", "—", "—", "✓"],
    "flujo LBM (mezcla, molienda)":           ["—", "—", "~", "✓"],
    "píloro abierto en el dominio":           ["—", "—", "—", "✓"],
}
tabla = {}
for v in ["1a", "1b", "4", "6"]:
    col = {}
    for met in OBJ:
        f = COMP[(COMP["versión"] == v) & (COMP["métrica"] == met)].iloc[0]
        col[met] = "—" if pd.isna(f["nominal"]) else f"{f['nominal']:.1f}  {'✅' if f['dentro'] else '❌'}"
    sub = COMP[(COMP["versión"] == v) & COMP["nominal"].notna()]
    col["métricas dentro del rango"] = f"{int(sub['dentro'].astype(float).sum())} / {len(sub)}"
    col["error de modelo normalizado (media)"] = f"{sub['error_modelo_norm'].mean():.3f}"
    col["error de modelo (peor métrica)"] = f"{sub['error_modelo_norm'].max():.3f}"
    for k, vals in FISICA.items():
        col[k] = vals[["1a", "1b", "4", "6"].index(v)]
    tabla[v] = col
TABLA = pd.DataFrame(tabla)
TABLA.columns = ["1a · química 0D", "1b · + tampón en equilibrio", "4 · + partículas y criba", "6 · integrado"]
rangos = {met: f"{lo}–{hi}" for met, (lo, hi) in OBJ.items()}
TABLA.insert(0, "rango medido", [rangos.get(i, "") for i in TABLA.index])
def estilo(df):
    s = df.style.set_properties(**{"text-align": "center"})
    s = s.set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
    s = s.apply(lambda c: ["background-color: #eef6fd" if i in ("métricas dentro del rango", "error de modelo normalizado (media)", "error de modelo (peor métrica)") else "" for i in df.index], axis=0)
    return s
display(estilo(TABLA))
TABLA.to_csv("figuras/tabla_comparativa.csv")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1.5, 1]})
a = axs[0]
mets = list(OBJ); versiones = ["1a", "1b", "4", "6"]
ancho = 0.2
for j, v in enumerate(versiones):
    vals = [COMP[(COMP["versión"] == v) & (COMP["métrica"] == m)]["error_modelo_norm"].iloc[0] for m in mets]
    xs = np.arange(len(mets)) + (j - 1.5) * ancho
    for x_, val in zip(xs, vals):
        if np.isnan(val):
            a.text(x_, 0.02, "—", ha="center", fontsize=9, color=GRIS_MEDIO)
        elif val == 0:
            a.plot(x_, 0.012, marker="v", color=COL_V[v], ms=6)
        else:
            a.bar(x_, val, width=ancho * 0.9, color=COL_V[v])
a.legend(handles=[Patch(color=COL_V[v], label=f"versión {v}") for v in versiones], fontsize=9)
a.set_xticks(range(len(mets))); a.set_xticklabels([m.replace(" [", "\n[") for m in mets], fontsize=9)
a.set(ylabel="error de modelo / anchura del rango", title="Error de modelo por métrica (▼ = dentro del rango, — = no calculable)")
a.grid(axis="x", visible=False)
a = axs[1]
med = [COMP[(COMP["versión"] == v)]["error_modelo_norm"].mean() for v in versiones]
dentro = [int(COMP[(COMP["versión"] == v)]["dentro"].astype(float).fillna(0).sum()) for v in versiones]
total = [int(COMP[(COMP["versión"] == v)]["nominal"].notna().sum()) for v in versiones]
barras = a.bar(versiones, med, color=[COL_V[v] for v in versiones])
for b_, d_, t_ in zip(barras, dentro, total):
    a.text(b_.get_x() + b_.get_width() / 2, b_.get_height() + 0.01, f"{d_}/{t_} dentro", ha="center", fontsize=9.5)
a.set(xlabel="versión del modelo", ylabel="error de modelo normalizado medio", title="Resumen: al integrar, el error baja")
a.grid(axis="x", visible=False)
fig.tight_layout(); fig.savefig("figuras/05_comparacion.png", dpi=200, bbox_inches="tight"); plt.show()

### 6.1 Registro de validación
Añadimos las filas de la versión 6, con el mismo formato que las de las fases 1b y 4.

In [ ]:
reg6 = []
for met, (lo, hi) in OBJ.items():
    f = COMP[(COMP["versión"] == "6") & (COMP["métrica"] == met)].iloc[0]
    reg6.append({"versión": "6 · integrado", "comida": "consenso" if "retención" in met else "1000 kcal", "métrica": met,
                 "nominal": f["nominal"], "p5": f["p5"], "p95": f["p95"], "objetivo_min": lo, "objetivo_max": hi,
                 "error_numérico": f["error_numérico"], "error_paramétrico": (f["p95"] - f["p5"]) / 2,
                 "error_de_modelo": disc(f["nominal"], lo, hi), "banda_solapa_dato": bool(f["p95"] >= lo and f["p5"] <= hi),
                 "fase": "6", "fecha": pd.Timestamp.now().strftime("%Y-%m-%d")})
reg6 = pd.DataFrame(reg6)
if os.path.exists("registro_validacion.csv"):
    previo = pd.read_csv("registro_validacion.csv")
    reg6 = pd.concat([previo[previo["versión"] != "6 · integrado"], reg6], ignore_index=True)
reg6.to_csv("registro_validacion.csv", index=False)
print(f"registro_validacion.csv: {len(reg6)} filas")

## 7. Resumen de verificación

In [ ]:
ver = pd.DataFrame(VERIFICACION)
ver["resultado"] = np.where(ver["pasa"], "✅ pasa", "❌ FALLA")
display(ver.drop(columns="pasa").style.format({"valor": "{:.3g}"}).hide(axis="index"))
assert ver["pasa"].all(), "Hay comprobaciones que no pasan"

## 8. Conclusiones

**Qué ha aportado integrar**
- **Es la versión con menos error de modelo y la única que pasa a la vez las métricas de vaciado y las de pH** (4 de 5 dentro del rango; véase la tabla). El tampón en equilibrio (1b) arregló el pH y las partículas con criba (4) arreglaron la forma del vaciado. La integración conserva ambas mejoras y añade pH, pepsina y ablandamiento **locales**.
- **La mejora no es uniforme.** En la retención a 2 h, la versión 4 (24 %) se acercaba algo más al rango que la integrada (≈ 22 %). Integrar añade física, pero también cierres nuevos con incertidumbre propia, y la tabla lo muestra sin esconderlo.
- **Aparecen fenómenos que los modelos 0D no podían tener**: la capa ácida junto a las paredes del cuerpo; la comida que espera tamponada en el centro; las partículas que avanzan pegadas a la pared y se trituran en el antro; y los péptidos que salen por el píloro con el flujo neto.

**Lo que sigue abierto (honestamente)**
- **La retención a 2 h** es la métrica más sensible y queda cerca del límite inferior del rango normal. Depende de cómo avanzan los sólidos sedimentados hacia el antro, que aquí se representa con el parámetro $\gamma$. Sus dos límites, partículas arrastradas o no por la pared que se contrae, dan resultados muy distintos. Lo vimos al construir el modelo y por eso usamos el punto medio y recorremos todo el rango en el conjunto. Es el **siguiente parámetro que conviene medir**, por ejemplo con imágenes de resonancia de la distribución intragástrica de sólidos.
- **El promediado en fase** subestima algo la dispersión del LBM directo (V1). Lo incluimos como incertidumbre (`escala_D`), pero un acoplamiento directo LBM–transporte en GPU durante algunos minutos de digestión permitiría cuantificarlo mejor.
- **La banda paramétrica** sale de un conjunto pequeño. En GPU, o con más tiempo de CPU, conviene ampliarlo.